# 🌳 Tree of Thought + Subquestion Decomposition

**Exercise duration: ~8 minutes** | Run every cell top-to-bottom

---

## What You'll Build

| Technique | Idea | What students should notice |
|---|---|---|
| **Tree of Thought (ToT)** | Explore multiple candidate next steps, score them, keep the best | A structured search can produce a stronger plan before final generation |
| **Subquestion Decomposition** | Break one hard problem into smaller answerable pieces | Better structure often improves reliability on multi-step math |
| **Plan-and-Solve** | Sketch a plan before executing | Separating planning from solving reduces messy reasoning |

---

**[RUN]** = just execute. **[TODO]** = fill in before running.

> ⚠️ Use a **GPU runtime**, e.g. T4/L4.

## 1. Setup

**[RUN]**

In [ ]:
#@title Install Dependencies {display-mode: "form"}
#@markdown Run this cell to install the required packages.
!pip install -q transformers accelerate datasets
import torch
print(f"\u2705 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(10)+chr(10)+'  WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU.'}")

In [ ]:
#@title Load the Model and Tokenizer {display-mode: "form"}
#@markdown Loads **Qwen2.5-0.5B-Instruct** and its tokenizer. This may take a few minutes on the first run.
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # half-precision: 2x faster, half the VRAM
    device_map="auto",           # auto-selects GPU
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
#@title Utility and Helper Functions {display-mode: "form"}
#@markdown Helper functions to generate model response and display results.
import re
import uuid, time, html as html_lib
from IPython.display import display, HTML


def clean_model_text(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()

    # Unwrap common LaTeX answer wrappers.
    # Examples: "\\boxed{42}", "$\\boxed{42}$", "boxed{42}", "$42$", "\\(42\\)", "\\[42\\]".
    t = re.sub(r"\$?\\\\boxed\{([^}]*)\}\$?", r"\1", t)
    t = re.sub(r"\$?boxed\{([^}]*)\}\$?", r"\1", t)

    m = re.fullmatch(r"\$([^$]+)\$", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\((.*)\\\)", t)
    if m:
        t = m.group(1).strip()

    m = re.fullmatch(r"\\\[(.*)\\\]", t)
    if m:
        t = m.group(1).strip()

    return t


def generate_response(messages, max_new_tokens=None, creative=False):
    """Send chat-formatted messages to the model and return the response."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = dict(
        do_sample=creative,
        temperature=0.7 if creative else 1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    if max_new_tokens is not None:
        gen_kwargs["max_new_tokens"] = max_new_tokens

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return clean_model_text(response)


def display_sample(question, answer):
    display(HTML(
        f"<p><strong>Question:</strong> {question}</p>"
        f"<p><strong>Answer:</strong> {answer}</p>"
    ))


def display_response(prompt_text, response_text, elapsed=None):
    uid = str(uuid.uuid4()).replace("-", "")
    time_tag = f"<div style='color:#666;font-size:90%;margin-top:4px'>&#x23F1; {elapsed:.1f}s</div>" if elapsed else ""
    safe_p = html_lib.escape(str(prompt_text).strip())
    safe_r = html_lib.escape(clean_model_text(str(response_text))).strip()
    display(HTML(
        "<style>.rt{border-collapse:collapse;width:100%;margin:10px 0;font-family:'Segoe UI',sans-serif;font-size:14px}"
        ".rt th,.rt td{border:1px solid #ddd;padding:9px 12px;vertical-align:top}"
        ".rt th{background:#f5f5f5;width:120px;font-weight:600}"
        "pre.rm{white-space:pre-wrap;margin:0}</style>"
        f"<table class='rt'>"
        f"<tr><th>Prompt</th><td><pre class='rm'>{safe_p}</pre></td></tr>"
        f"<tr><th>Response</th><td><pre class='rm'>{safe_r}</pre>{time_tag}</td></tr>"
        "</table>"
    ))


def run_and_display(messages, max_new_tokens=None, creative=False):
    """Generate and display in a table. Returns the response string."""
    prompt_text = messages[-1]["content"]
    t0 = time.time()
    response = generate_response(messages, max_new_tokens=max_new_tokens, creative=creative)
    elapsed = time.time() - t0
    display_response(prompt_text, response, elapsed)
    return response

## 🌳 Part 1: Tree of Thought (ToT)

### What's different about Tree of Thought?

Standard Chain of Thought forces the model down a **single reasoning path**. But what if the first step it takes is suboptimal?

**Tree of Thought** lets the model branch out 🌳. This is what it does:

1. Start with an empty outline
2. Generate **multiple candidate next outline lines**
3. Score them
4. Keep only the best **beams**
5. Repeat for a few steps
6. Use the best outline to write the final story

So your task here is:

> **First build a short outline through search, then write the story from the best outline.**

That is the whole point of ToT: **don't commit too early**. It's essentially **beam search applied to reasoning**.

Think of it like brainstorming: instead of committing to the first idea, you generate several options, compare them, and develop the most promising one.

Each section below walks through one stage of this process — from setup 🛠️ to proposal 💡 to final composition.

### Step 1: Configuration & Prompt Templates

**[RUN]** We define two key parameters:
- **B** (branching factor): how many candidate ideas to generate at each step
- **D** (depth): how many reasoning steps to take

🧮 Wider B → more ideas explored; deeper D → longer reasoning chains.

In [ ]:
from typing import List, Tuple, Dict
import random
random.seed(42)

B = 3   # keep top 3 beams after each step
D = 2   # build a 2-line outline before composing the story

**[RUN]** Define the prompt templates. Notice the `{placeholders}`.

In [ ]:
TASK_INSTRUCTION = (
    "Write a vivid short story of about 100 words for a general audience. "
    "It must include: (1) strong imagery, (2) a surprising twist, and (3) an emotionally satisfying ending. "
    "Do not use proper names."
)

PROPOSE_PROMPT = """
You are planning a short story.

Story brief:
{task}

Current outline:
{outline}

Write exactly ONE good next outline line for the story.
It should move the story forward in a concrete way.
Return ONLY that one line.
""".strip()

SCORE_PROMPT = """
You are evaluating one candidate outline line for a short story.

Story brief:
{task}

Current outline:
{outline}

Candidate next line:
{candidate}

Give one integer score from 0 to 10 based on:
- fit with the brief
- coherence with the current outline
- creativity / story potential

Return ONLY the integer.
""".strip()

### Quick Python check: `.format()` fills placeholders

**[RUN]**

In [ ]:
# How .format() fills placeholders:
prompt = PROPOSE_PROMPT.format(task=TASK_INSTRUCTION, outline="(none yet)")
print(prompt)

### ✏️ [TODO] Try it for the scoring template

`SCORE_PROMPT` has **three** placeholders: `{task}`, `{outline}`, `{candidate}`.

This is exactly how the tree gets generated:
- the current outline is inserted into the prompt,
- the model proposes the **next** line,
- then that candidate line is scored.

In [ ]:
#TODO: Call SCORE_PROMPT_TEMPLATE.format() with three values and print the result

Good. You'll use this `.format()` pattern in every exercise below.

### Step 2: ToT Core Functions (propose → score)

**[IMPORTANT]** Before running the next cell, read this note on **Chat Templates**:

When using a modern instruction-tuned model like Qwen, you don't just send a raw string. Instead, messages are formatted as a list of dictionaries, each with:
- **`role`**: either `"user"` or `"assistant"`
- **`content`**: the actual text

```python
message = [{"role": "user", "content": "Your prompt here"}]
```

This format is called the **Chat Template** and it's how the model knows who is speaking.


**Speed tip built into `generate_response`:**
- `creative=False` → greedy, fastest. Use for scoring (just need an integer).
- `creative=True` → sampled. Use for story generation.

Below, the tree is generated very simply:

- `propose_next_beams(...)` makes **`b` separate model calls** to get `b` different candidate next lines
- `score_beams(...)` scores each candidate
- later, beam search keeps only the best few

This makes the search visible and easy to explain in class.

### 🎯 [TODO] Implement `score_candidates`

In `score_candidates`: build the prompt using `SCORE_PROMPT_TEMPLATE`, create the message dict, and call the model.

In [ ]:
def propose_next_beams(outline_lines, b):
    """Generate b candidate next outline lines for the current partial outline."""
    outline_text = "\n".join(f"- {line}" for line in outline_lines) if outline_lines else "(empty outline)"
    prompt = PROPOSE_PROMPT.format(task=TASK_INSTRUCTION, outline=outline_text)
    message = [{"role": "user", "content": prompt}]

    candidates = []
    for _ in range(b):
        response = generate_response(message, max_new_tokens=60, creative=True)
        response = clean_model_text(response).split("\n")[0].strip("-• ").strip()
        if response and response not in candidates:
            candidates.append(response)

    return candidates


def score_beams(outline_lines, candidates):
    """Score each candidate next line. Returns list of (candidate, score)."""
    scored = []
    outline_text = "\n".join(f"- {line}" for line in outline_lines) if outline_lines else "(empty outline)"

    for candidate in candidates:
       # TODO: create the prompt using SCORE_PROMPT_TEMPLATE and .format()
        prompt = ""
        # TODO: create the message to be sent to the model using the Chat Template method explained above
        message = []
        try:
            resp = generate_response(message, max_new_tokens=6, creative=False)
            m = re.search(r"\d+", resp)
            score = float(m.group()) if m else 0.0
            score = max(0.0, min(10.0, score))
        except Exception:
            score = 0.0

        scored.append((candidate, score))

    return scored

### 🎯 [TODO] Step 3: Beam Search over Outline Steps

This is the core of the Tree-of-Thought algorithm 🌳. It loops through reasoning steps at every depth:

1. **Propose** candidate thoughts using `propose_next_beams`
2. **Score & select** the top `B` outlines using `score_candidates`
3. **Expand** further until the outline is complete

Your task: fill in the two `TODO` lines to call the right functions.

In [ ]:
def beam_search_outline(b=B, d=D):
    """
    Tree-of-Thought beam search over short story outlines.

    Beam state:
        {
            "outline": [...],
            "avg_score": float
        }
    """
    beams = [{"outline": [], "avg_score": 0.0}]
    history = []

    for step in range(1, d + 1):
        expanded = []

        for beam in beams:
            outline = beam["outline"]
            candidate_prompts = # TODO: call propose_next_beams with the current beams and b
            scored = # TODO: call score_candidates with the current beams and candidate_prompts

            for candidate, score in scored:
                new_outline = outline + [candidate]
                new_avg = (beam["avg_score"] * (step - 1) + score) / step
                expanded.append({
                    "outline": new_outline,
                    "avg_score": new_avg,
                    "new_line": candidate,
                    "step_score": score,
                    "parent_outline": outline,
                })

        expanded.sort(key=lambda x: x["avg_score"], reverse=True)
        kept = expanded[:b]

        history.append({
            "step": step,
            "expanded": expanded,
            "kept": kept,
        })

        print(f"Step {step}: explored {len(expanded)} candidates, kept top {len(kept)} beams")
        for rank, beam in enumerate(kept, 1):
            print(f"  #{rank} | avg={beam['avg_score']:.1f} | added: {beam['new_line']}")

        beams = [{"outline": beam["outline"], "avg_score": beam["avg_score"]} for beam in kept]

    return beams[0]["outline"], history

### 🎯 [TODO] Step 4: Compose the Final Story

Now we use the **best outline found by search** and ask the model to write the final story from it.

That makes the workflow explicit:

**search first → generate final answer second**

Complete the two `TODO` lines below:

In [ ]:
COMPOSE_PROMPT = """
Using the outline below, write the final story in about 100 words.

Story brief:
{task}

Outline:
{beats}

Write one polished paragraph.
""".strip()

def compose_story(beams):
    beats_text = "\n".join(f"- {b}" for b in beams)
    # TODO: create the prompt using COMPOSE_PROMPT_TEMPLATE and .format()
    prompt = ""
    # TODO: create the message with this prompt
    message = []
    return run_and_display(message, max_new_tokens=180, creative=True)

### Step 5: Run the Full ToT Pipeline

This cell now shows:
- how the outline starts from an **empty** state,
- which candidates were explored,
- which beams survived,
- and the final story written from the best outline.

In [ ]:
print("Running Tree-of-Thought outline search...\n")
best_beams, tot_history = beam_search_outline(b=B, d=D)

print("\nBest outline selected by search:")
for i, line in enumerate(best_beams, 1):
    print(f"  {i}. {line}")

print("\nComposing final story from the best outline...\n")
tot_story = compose_story(best_beams)

### Baseline: Direct Story (No ToT)

**[RUN]** Same story task, but this time the model writes immediately.

Now students can compare:

- **Direct generation**: one shot
- **ToT generation**: search over multiple candidate outline paths first

In [ ]:
BASELINE_PROMPT = """
{task}

Write the story now in about 100 words as one polished paragraph.
""".strip()

print("Zero-shot baseline (no Tree of Thought)...\n")
baseline_prompt = BASELINE_PROMPT.format(task=TASK_INSTRUCTION)
baseline_story = run_and_display([{"role": "user", "content": baseline_prompt}], max_new_tokens=180, creative=True)

### Optional: Quick side-by-side quality check

The same model can act as a **rough judge** here.
That is not a rigorous benchmark, but it helps make the classroom point visible:

- the baseline writes immediately,
- ToT first searches over several outline paths,
- then writes from the strongest outline.

In [ ]:
JUDGE_PROMPT = """
You are comparing two short stories written for the same brief.

Brief:
{task}

Story A:
{story_a}

Story B:
{story_b}

For each story, give an integer 0-10 for:
- imagery
- twist quality
- satisfying ending
- overall fit to the brief

Then say which story is better overall.

Use this exact format:
A: <score>
B: <score>
""".strip()

judge_text = run_and_display(
    [{"role": "user", "content": JUDGE_PROMPT.format(
        task=TASK_INSTRUCTION,
        story_a=baseline_story,
        story_b=tot_story
    )}],
    max_new_tokens=120,
    creative=False
)

print("Interpretation:")
print("A = direct baseline, B = Tree-of-Thought story")

---

## 🧠 Part 2: Subquestion Decomposition

### The big idea

Sometimes a problem is just too complex to solve in one shot 🧠💥. Instead of attacking it head-on, we break it into smaller, focused sub-questions — each one answerable on its own — and combine the answers at the end.

This mimics how humans tackle hard problems: **think → split → solve → combine** 🔄

We'll test this on the **GSM8K socratic split**, which includes worked examples with explicit subquestion breakdowns.

Furthermore, we compare **direct solving** vs **subquestion decomposition** in a much neater way.

In [ ]:
#@title Load the Socratic GSM8K Split {display-mode: "form"}
#@markdown Loads the socratic version of GSM8K, where each answer is broken into guided sub-questions.
ds_socratic = load_dataset("openai/gsm8k", "socratic")
train_split_socratic = ds_socratic["train"]
test_split_socratic  = ds_socratic["test"]

print(f"Train: {len(train_split_socratic)} | Test: {len(test_split_socratic)}")


In [ ]:
#@title Explore a Socratic Sample {display-mode: "form"}
#@markdown Displays the first question-answer pair from the socratic training split.

display_sample(train_split_socratic[0]["question"], train_split_socratic[0]["answer"])

In [ ]:
#@title Utility functions {display-mode: "form"}
#@markdown Functions to extract only the numerical answers from the true values and predictions

def extract_gold_answer(answer_text):
    parts = str(answer_text).split("####")
    return parts[-1].strip() if len(parts) > 1 else answer_text.strip()

def extract_pred_answer(text):
    text = clean_model_text(text)
    patterns = [
        r"Final answer\s*[:\-]\s*([^\n]+)",
        r"Answer\s*[:\-]\s*([^\n]+)",
        r"Therefore[^\n]*?([\$]?[\d,]+(?:\.\d+)?)",
        r"([\$]?[\d,]+(?:\.\d+)?)\s*$",
    ]
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip().replace(",", "")
    return text.strip().split("\n")[-1].strip()

### 🎯 [TODO] Build two prompts for the same question

We will compare:

- **DIRECT_PROMPT**: solve the problem in one go
- **SUBQUESTION_PROMPT**: explicitly break it into a few small questions first

Draft these 2 prompts below such that the model can efficiently follow this strategy.

In [ ]:
DIRECT_PROMPT = """

""".strip() #

SUBQUESTION_PROMPT = """

""".strip()

In [ ]:
#@title Comparison functions {display-mode: "form"}
#@markdown Functions to compare the Direct and Subquestion prompts
def run_prompt(prompt_text, max_new_tokens=256):
    return generate_response([{"role": "user", "content": prompt_text}], max_new_tokens=max_new_tokens, creative=False)

def compare_on_example(example_idx=0):
    item = test_split_socratic[example_idx]
    question = item["question"]
    gold = extract_gold_answer(item["answer"])

    direct_text = run_prompt(DIRECT_PROMPT.format(question=question))
    subq_text = run_prompt(SUBQUESTION_PROMPT.format(question=question))

    direct_pred = extract_pred_answer(direct_text)
    subq_pred = extract_pred_answer(subq_text)

    display(HTML(
        f"<h4>Question</h4><p>{html_lib.escape(question)}</p>"
        f"<p><strong>Gold answer:</strong> {html_lib.escape(gold)}</p>"
    ))

    display_response("Direct baseline prompt", direct_text)
    display_response("Subquestion decomposition prompt", subq_text)

    return {
        "question": question,
        "gold": gold,
        "direct_pred": direct_pred,
        "subq_pred": subq_pred,
        "direct_correct": direct_pred == gold,
        "subq_correct": subq_pred == gold,
    }

### Example comparison on one test problem

**[RUN]** Start with one problem so students can inspect the two outputs side by side.

In [ ]:
#@title Run the comparison {display-mode: "form"}
#@markdown Call the function to compare the approaches on one data sample

subq_example_result = compare_on_example(example_idx=3)

### Small mini-benchmark (3 problems)

One example can be noisy.  
So let's compare both strategies on a tiny set of 3 test problems and summarize the results in a table.

In [ ]:
sample_indices = [1, 3, 10] #feel free to choose any other indices. There are 7473 samples in train and 1319 in test set
rows = []

for idx in sample_indices:
    result = compare_on_example(example_idx=idx)
    rows.append(result)

direct_acc = sum(r["direct_correct"] for r in rows)
subq_acc = sum(r["subq_correct"] for r in rows)

summary_rows = []
for i, r in zip(sample_indices, rows):
    summary_rows.append(
        f"<tr><td>{i}</td>"
        f"<td>{'✅' if r['direct_correct'] else '❌'}</td>"
        f"<td>{'✅' if r['subq_correct'] else '❌'}</td>"
        f"<td>{html_lib.escape(r['gold'])}</td></tr>"
    )

---

## 🗺️ Plan-and-Solve Prompting

### What makes this different?

Plan-and-Solve explicitly separates the *planning* phase from the *execution* phase — just like how you'd sketch a rough approach on a whiteboard before diving into code.

**How it works:**
1. **Plan Phase** 🗺️ — The model outlines what steps are needed: "First identify what's asked → then gather given info → compute step by step."
2. **Solve Phase** 🧮 — The model follows its own plan systematically to reach the final answer.


This is similar to subquestioning, but the structure is a **plan**, not a tree of sub-questions. It prevents the model from rushing to compute before it fully understands the problem structure.

In [ ]:
test_q = test_split_socratic[0]["question"]

few_shot_PS = (
    "Follow this format to solve the question:\n\n"
    "Q: James runs 3 sprints 3 times a week, 60 meters each. Total meters per week?\n"
    "A:\n"
    "Given: 3 sprints x 3 times = 9 sprints per week; each sprint is 60 meters.\n"
    "Plan: Multiply total sprints by distance per sprint.\n"
    "Solve: 9 x 60 = 540.\n"
    "Final answer: 540\n\n"
    f"Q: {test_q}\nA:"
)
print(few_shot_PS)
run_and_display([{"role": "user", "content": few_shot_PS}], max_new_tokens=512)

### 🎯 [TODO] Zero-shot Plan-and-Solve

Now write your own zero-shot instruction that teaches the model to plan before solving. Your instruction should direct the model to:

- **Outline what's given**
- **Plan**
- **Solve**
- **Final answer**

In [ ]:
test_q = test_split_socratic[0]["question"]

prompt = f"""

""".strip()

run_and_display([{"role": "user", "content": prompt}], max_new_tokens=512)

# 🏁 Key Takeaways

1. **Tree of Thought is search, not magic.**  
   It works by exploring multiple candidate next steps, scoring them, and keeping the best beams.

2. **The initial outline starts empty on purpose.**  
   The first generation happens when the model is prompted with the empty outline and asked for the **next** line.

3. **Subquestion decomposition is easiest to appreciate when compared against a direct baseline.**  
   A neat side-by-side table makes the benefit much clearer than a single long raw output.

4. **Structure helps students see the point.**  
   The goal is not just better answers, but making the reasoning process visible and inspectable.